In [9]:
from pathlib import Path

import geopandas as gpd
import polars as pl
import polars_h3 as plh3
import yaml
from detect_home import detect_home_work, map_to_settlements

with open("../config.copenhagen_sample.yaml") as fp:
    config = yaml.safe_load(fp)

In [11]:
PARTITION = 0
SETTLEMENTS_SHAPEFILE = "../data/osm/hungary/settlements.geojson"
SETTLEMENTS_SHAPEFILE = None
DIR = f"{config['dirs']['output']}/{config['city']}"

In [3]:
data = pl.read_parquet(f"{DIR}/stops/{PARTITION:02x}.parquet")
data.head(3)

device_id,stop_id,interval_start,interval_stop,interval_lenght,latitude_mean,longitude_mean,latitude_std,longitude_std,number_of_pings,place_label
i64,i32,"datetime[μs, Europe/Budapest]","datetime[μs, Europe/Budapest]",duration[μs],f64,f64,f64,f64,u32,i64
4,1,2025-04-07 06:03:13 CEST,2025-04-12 17:39:15 CEST,5d 11h 36m 2s,55.718058,12.465532,13.0,7.0,37,1
4,2,2025-04-12 18:04:34 CEST,2025-04-12 23:37:00 CEST,5h 32m 26s,55.686993,12.532105,7.0,9.0,162,2
4,3,2025-04-13 00:08:46 CEST,2025-04-13 01:53:55 CEST,1h 45m 9s,55.717978,12.46554,25.0,11.0,64,1


In [4]:
print(data.head(3).to_pandas().to_markdown(index=False))

|   device_id |   stop_id | interval_start            | interval_stop             | interval_lenght   |   latitude_mean |   longitude_mean |   latitude_std |   longitude_std |   number_of_pings |   place_label |
|------------:|----------:|:--------------------------|:--------------------------|:------------------|----------------:|-----------------:|---------------:|----------------:|------------------:|--------------:|
|           4 |         1 | 2025-04-07 06:03:13+02:00 | 2025-04-12 17:39:15+02:00 | 5 days 11:36:02   |         55.7181 |          12.4655 |             13 |               7 |                37 |             1 |
|           4 |         2 | 2025-04-12 18:04:34+02:00 | 2025-04-12 23:37:00+02:00 | 0 days 05:32:26   |         55.687  |          12.5321 |              7 |               9 |               162 |             2 |
|           4 |         3 | 2025-04-13 00:08:46+02:00 | 2025-04-13 01:53:55+02:00 | 0 days 01:45:09   |         55.718  |          12.4655 |            

In [5]:
data = data.with_columns(
    plh3.int_to_str(
        plh3.latlng_to_cell(pl.col("latitude_mean"), pl.col("longitude_mean"), 8)
    ).alias("h3_8"),
    plh3.int_to_str(
        plh3.latlng_to_cell(pl.col("latitude_mean"), pl.col("longitude_mean"), 9)
    ).alias("h3_9"),
    plh3.int_to_str(
        plh3.latlng_to_cell(pl.col("latitude_mean"), pl.col("longitude_mean"), 10)
    ).alias("h3_10"),
    pl.col("interval_start").dt.date().alias("day"),
    pl.col("interval_start").dt.time().alias("time"),
    pl.col("interval_start").dt.weekday().alias("dow"),
)

In [6]:
data.head(3)

device_id,stop_id,interval_start,interval_stop,interval_lenght,latitude_mean,longitude_mean,latitude_std,longitude_std,number_of_pings,place_label,h3_8,h3_9,h3_10,day,time,dow
i64,i32,"datetime[μs, Europe/Budapest]","datetime[μs, Europe/Budapest]",duration[μs],f64,f64,f64,f64,u32,i64,str,str,str,date,time,i8
4,1,2025-04-07 06:03:13 CEST,2025-04-12 17:39:15 CEST,5d 11h 36m 2s,55.718058,12.465532,13.0,7.0,37,1,"""881f0591e5fffff""","""891f0591e4fffff""","""8a1f0591e4e7fff""",2025-04-07,06:03:13,1
4,2,2025-04-12 18:04:34 CEST,2025-04-12 23:37:00 CEST,5h 32m 26s,55.686993,12.532105,7.0,9.0,162,2,"""881f058309fffff""","""891f0583083ffff""","""8a1f0583082ffff""",2025-04-12,18:04:34,6
4,3,2025-04-13 00:08:46 CEST,2025-04-13 01:53:55 CEST,1h 45m 9s,55.717978,12.46554,25.0,11.0,64,1,"""881f0591e5fffff""","""891f0591e4fffff""","""8a1f0591e4e7fff""",2025-04-13,00:08:46,7


In [7]:
with open("06_home_detection_config.yaml") as fp:
    hd_config = yaml.safe_load(fp)

In [10]:
Path(f"{DIR}/home_work").mkdir(parents=True, exist_ok=True)
hw8 = detect_home_work(data, hd_config, spatial_id="h3_8", device_id="device_id")
hw8.write_csv(f"{DIR}/home_work/home_work_{PARTITION:02x}_h3_8.csv")
hw9 = detect_home_work(data, hd_config, spatial_id="h3_9", device_id="device_id")
hw9.write_csv(f"{DIR}/home_work/home_work_{PARTITION:02x}_h3_9.csv")
hw10 = detect_home_work(data, hd_config, spatial_id="h3_10", device_id="device_id")
hw10.write_csv(f"{DIR}/home_work/home_work_{PARTITION:02x}_h3_10.csv")

Detect home in settlements

In [12]:
if SETTLEMENTS_SHAPEFILE:
    settlements = gpd.read_file(SETTLEMENTS_SHAPEFILE).set_crs(4326)
    settlements = settlements[settlements["name"] != "Budapest"].copy()

    # runs 3s for Budapest p00
    data_s = map_to_settlements(data, settlements)

    hws = detect_home_work(data_s, hd_config, spatial_id="name", device_id="device_id")
    hws.write_csv(f"{DIR}/home_work/home_work_{PARTITION:02x}_settlement.csv")